In [1]:
from transformers import AutoTokenizer, AutoModel
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to("cuda")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [ ]:
import pandas as pd
import numpy as np
import warnings
import re
import torch

warnings.filterwarnings("ignore")


In [3]:
df = pd.read_csv(r'reddit_dataset.csv')
df.drop(columns = ['row_id','subreddit'], inplace=True)

In [4]:
y_labels_original = df[['rule_violation']]
x = df.drop(columns = ['rule_violation'])

In [5]:
def cleaning_text(text_file):

     ## Removing URLs
    pattern = re.compile('https?:\/\/\S+|www\.\S+|Https?:\/\/\S+|\S+\.com\S+|\S+\.com|\[.*?\]|\S+ \. com.*')
    for i in range(len(text_file)):
        text_file[i] = pattern.sub(r'',text_file[i])

    ##Removing HTML rags
    pattern = re.compile('<.*?>')
    for i in range(len(text_file)):
        text_file[i] = pattern.sub(r'',text_file[i])

    ## Removing Emails and Hashtags
    pattern = re.compile('#\S+|@\S+|\S+\@\S+|\S+@')
    for i in range(len(text_file)):
        text_file[i] = pattern.sub(r'',text_file[i])

    ### Removing username and subreddit mentions
    pattern = re.compile('u\/\S+|r\/\S+')
    for i in range(len(text_file)):
        text_file[i] = pattern.sub(r'',text_file[i])

    #emotions, symbols, pictographs, transport and map symbols, flags etx.
    pattern = re.compile("["
                            u"\U0001F600-\U0001F64F"  # emoticons
                            u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                            u"\U0001F680-\U0001F6FF"  # transport & map symbols
                            u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                            u"\U00002702-\U000027B0"
                            u"\U000024C2-\U0001F251"
                            "]+", flags=re.UNICODE)
    for i in range(len(text_file)):
        text_file[i] = pattern.sub(r'',text_file[i])

    ##Removing Numbers & \n spaces
    pattern = re.compile('\d|\\n')
    for i in range(len(text_file)):
        text_file[i] = pattern.sub(r'',text_file[i])

    return text_file

##TRAINING DATA ----------------------
x['body'] = cleaning_text(list(x['body']))
x['positive_example_1'] = cleaning_text(list(x['positive_example_1']))
x['positive_example_2'] = cleaning_text(list(x['positive_example_2']))
x['negative_example_1'] = cleaning_text(list(x['negative_example_1']))
x['negative_example_2'] = cleaning_text(list(x['negative_example_2']))

In [6]:
## Made the text into tokens
## Made the tokens into embeddings.
##Used batches to convert tokens.

def embedded_text(df_col, model, tokenizer, batch_size = 32):
  df_col_list = df_col.to_list()
  tokenized_text = tokenizer(df_col_list, padding = True, truncation = True, return_tensors = 'pt')

  tokenized_text = tokenized_text.to("cuda")
  embeddings = []
  for i in range(0, len(tokenized_text['input_ids']), batch_size):
    batch = {k: v[i:i+batch_size].to(model.device) for k, v in tokenized_text.items()}
    with torch.no_grad():
      outputs = model(**batch)
        # Use the embeddings of the first token (CLS token) as the sentence embedding
    embeddings.append(outputs.last_hidden_state[:, 0, :].cpu())
  troch_concatinated = torch.cat(embeddings, dim=0)
  df_col = [emb.tolist() for emb in troch_concatinated]
  return df_col

x['body'] = embedded_text(df_col = x['body'], model = model ,tokenizer=tokenizer, batch_size = 32)
x['positive_example_1'] = embedded_text(df_col = x['positive_example_1'], model = model ,tokenizer=tokenizer, batch_size = 32)
x['positive_example_2'] = embedded_text(df_col = x['positive_example_2'], model = model ,tokenizer=tokenizer, batch_size = 32)
x['negative_example_1'] = embedded_text(df_col = x['negative_example_1'], model = model ,tokenizer=tokenizer, batch_size = 32)
x['negative_example_2'] = embedded_text(df_col = x['negative_example_2'], model = model ,tokenizer=tokenizer, batch_size = 32)

In [7]:
def embedded_rule_text(df_col, model, tokenizer, batch_size = 32):
  df_col_list = df_col.to_list()
  tokenized_text = tokenizer(df_col_list, padding = True, truncation = True, return_tensors = 'pt')

  tokenized_text = tokenized_text.to("cuda")
  embeddings = []
  for i in range(0, len(tokenized_text['input_ids']), batch_size):
    batch = {k: v[i:i+batch_size].to(model.device) for k, v in tokenized_text.items()}
    with torch.no_grad():
      outputs = model(**batch)
        # Use the embeddings of the first token (CLS token) as the sentence embedding
    embeddings.append(outputs.last_hidden_state[:, 0, :].cpu())
  troch_concatinated = torch.cat(embeddings, dim=0)
  df_col = [emb.tolist() for emb in troch_concatinated]
  return df_col

x['rule']= embedded_rule_text(df_col = x['rule'], model = model ,tokenizer=tokenizer, batch_size = 32)

In [8]:
x_copy = x.copy()

### Apprach to use Rules and Examples as features with targets.

In [9]:
x_positive_1_df = x_copy[['rule', 'positive_example_1']].rename(columns={'positive_example_1': 'body'})
x_positive_2_df = x_copy[['rule', 'positive_example_2']].rename(columns={'positive_example_2': 'body'})
x_negative_1_df = x_copy[['rule', 'negative_example_1']].rename(columns={'negative_example_1': 'body'})
x_negative_2_df = x_copy[['rule', 'negative_example_2']].rename(columns={'negative_example_2': 'body'})
x_body_df = x_copy[['rule','body']]

In [10]:
x_positive_1_df['rule_violation'] = 1
x_positive_2_df['rule_violation'] = 1
x_negative_1_df['rule_violation'] = 0
x_negative_2_df['rule_violation'] = 0
x_body_df['rule_violation'] = y_labels_original['rule_violation']

In [11]:
x_pytorch_data = pd.concat([x_positive_1_df, x_positive_2_df, x_negative_1_df, x_negative_2_df, x_body_df], axis = 0).reset_index(drop=True)

In [12]:
# These variables are used for the PyTorch model.
x_rule_pt = x_pytorch_data['rule']
x_body_pt = x_pytorch_data['body']
y_pt = x_pytorch_data[['rule_violation']]

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F

In [14]:
X_train_b, X_val_b, X_train_r, X_val_r, y_train, y_val = train_test_split(
    x_body_pt, x_rule_pt, y_pt, test_size=0.2, random_state=42
)
# Convert y_train and y_val to Series of int64 to avoid type issues with CrossEntropyLoss
y_train = y_train.squeeze().astype(int)
y_val = y_val.squeeze().astype(int)
class ViolationDataset(Dataset):
    def __init__(self, body_emb, rule_emb, labels):
        self.body_emb = body_emb
        self.rule_emb = rule_emb
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Retrieve embeddings
        body_embedding = self.body_emb.iloc[idx]
        rule_embedding = self.rule_emb.iloc[idx]

        # Handle body_embedding: ensure it's a non-empty list, otherwise use a zero tensor
        if isinstance(body_embedding, list) and len(body_embedding) > 0:
            body_tensor = torch.tensor(body_embedding, dtype=torch.float32)
        else:
            # Covers [], None, np.nan, or other non-list/empty-list types
            body_tensor = torch.zeros(768, dtype=torch.float32)

        # Handle rule_embedding: ensure it's a non-empty list, otherwise use a zero tensor
        if isinstance(rule_embedding, list) and len(rule_embedding) > 0:
            rule_tensor = torch.tensor(rule_embedding, dtype=torch.float32)
        else:
            # Covers [], None, np.nan, or other non-list/empty-list types
            rule_tensor = torch.zeros(768, dtype=torch.float32)

        label_tensor = torch.tensor(self.labels.iloc[idx], dtype=torch.long)
        return body_tensor, rule_tensor, label_tensor

train_data = ViolationDataset(X_train_b, X_train_r, y_train)
val_data = ViolationDataset(X_val_b, X_val_r, y_val)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)

class ViolationClassifier(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=256, dropout=0.2):
        super(ViolationClassifier, self).__init__()

        # First fully connected layer
        self.fc1 = nn.Linear(input_dim * 2, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.dropout1 = nn.Dropout(dropout)

        # Second fully connected layer
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.bn2 = nn.BatchNorm1d(hidden_dim // 2)
        self.dropout2 = nn.Dropout(dropout)

        # Output layer
        self.fc_out = nn.Linear(hidden_dim // 2, 2)  # Binary classification output

    def forward(self, body_emb, rule_emb):
        x = torch.cat((body_emb, rule_emb), dim=1)  # Concatenate embeddings

        # Pass through the first layer with Batch Normalization, ReLU, and Dropout
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)

        # Pass through the second layer with Batch Normalization, ReLU, and Dropout
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)

        # Output layer
        out = self.fc_out(x)
        return out

# Check if CUDA is available and use GPU if it is
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ViolationClassifier(hidden_dim=256, dropout=0.5).to(device) # Initialize with new parameters

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
epochs = 25 # Define number of epochs

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for body_batch, rule_batch, y_batch in train_loader:
        # Move data to the same device as the model
        body_batch, rule_batch, y_batch = body_batch.to(device), rule_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(body_batch, rule_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    model.eval()
    correct, total = [], []
    with torch.no_grad():
        for body_batch, rule_batch, y_batch in val_loader:
            # Move data to the same device as the model
            body_batch, rule_batch, y_batch = body_batch.to(device), rule_batch.to(device), y_batch.to(device)

            outputs = model(body_batch, rule_batch)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == y_batch).sum().item()
            correct.extend(preds.)
            total += y_batch.size(0)
    acc = correct / total
    print(f"Epoch {epoch+1}: Loss={total_loss/len(train_loader):.4f}, Accuracy={acc:.4f}")

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

def print_classification_metrics(y_true, y_pred, y_proba=None):
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision:  {precision_score(y_true, y_pred):.4f}")
    print(f"Recall :     {recall_score(y_true, y_pred):.4f}")
    print(f"F1-score:   {f1_score(y_true, y_pred):.4f}")
    print(f"roc_auc_score:   {roc_auc_score(y_true, y_pred):.4f}")



In [53]:
with torch.no_grad():
  pred_data = []
  actual_data = []
  for body_batch, rule_batch, y_batch in train_loader:
    body_batch, rule_batch, y_batch = body_batch.to(device), rule_batch.to(device), y_batch.to(device)
    outputs = model(body_batch, rule_batch)
    outputs = torch.argmax(outputs, dim=1)
    pred_data.extend(outputs.cpu())
    actual_data.extend(y_batch.cpu())

y_pred = np.array(pred_data).reshape(-1,1)
y_actual = np.array(actual_data).reshape(-1,1)

print_classification_metrics(y_true = y_actual, y_pred = y_pred)

Accuracy: 0.9579
Precision:  0.9639
Recall :     0.9516
F1-score:   0.9577
roc_auc_score:   0.9579


In [54]:
with torch.no_grad():
  pred_data = []
  actual_data = []
  for body_batch, rule_batch, y_batch in val_loader:
    body_batch, rule_batch, y_batch = body_batch.to(device), rule_batch.to(device), y_batch.to(device)
    outputs = model(body_batch, rule_batch)
    outputs = torch.argmax(outputs, dim=1)
    pred_data.extend(outputs.cpu())
    actual_data.extend(y_batch.cpu())

y_pred = np.array(pred_data).reshape(-1,1)
y_actual = np.array(actual_data).reshape(-1,1)

print_classification_metrics(y_true = y_actual, y_pred = y_pred)

Accuracy: 0.9285
Precision:  0.9352
Recall :     0.9214
F1-score:   0.9283
roc_auc_score:   0.9286


In [ ]:
torch.save(model.state_dict(), "model.pth")